# Mi revisión de Bronze para automatización

Consulta de lectura del 12 de noviembre de 2026.
Reproduzco cuatro consultas verificadas mediante Genie.
No necesito ejecutar este notebook antes de cada carga.
Las consultas leen auditorías y catálogo; no escriben tablas.
Los nombres workspace.gtfs_bogota corresponden al entorno actual.
El notebook completo todavía no tiene una ejecución documentada.
El informe y los límites están en docs/bronze_review_2026_11_12.md.

## Bloque 1: Duplicación de auditoría

Nueve corridas. Dos contienen 57 copias adicionales en los campos comparados. No diagnostico duplicados de datos Bronze.
Consulta Genie: pub_toolu_bdrk_01KVaetkBiJQBnMyUoafiNYK_output.
Conservo el SQL ejecutado sin cambios.

In [ ]:
%sql
WITH group_stats AS (
  SELECT 
    `quality_run_id`,
    `check_name`,
    `entity_name`,
    `severity`,
    COUNT(*) as row_count,
    COUNT(DISTINCT STRUCT(`status`, `observed_value`, `expected_value`, `details`)) as distinct_variants
  FROM `workspace`.`gtfs_bogota`.`audit_gtfs_bronze_snapshot_quality`
  GROUP BY `quality_run_id`, `check_name`, `entity_name`, `severity`
),
run_summary AS (
  SELECT 
    `quality_run_id`,
    COUNT_IF(`row_count` > 1) as duplicate_groups,
    SUM(CASE WHEN `row_count` > 1 THEN `row_count` - 1 ELSE 0 END) as extra_copies,
    COUNT_IF(`distinct_variants` > 1) as groups_with_contradictory_variants
  FROM group_stats
  GROUP BY `quality_run_id`
)
SELECT 
  rs.`quality_run_id`,
  rs.`duplicate_groups`,
  rs.`extra_copies`,
  rs.`groups_with_contradictory_variants`,
  q.`checked_at_utc`
FROM run_summary rs
INNER JOIN (
  SELECT DISTINCT `quality_run_id`, `checked_at_utc`
  FROM `workspace`.`gtfs_bogota`.`audit_gtfs_bronze_snapshot_quality`
) q ON rs.`quality_run_id` = q.`quality_run_id`
ORDER BY q.`checked_at_utc` DESC
LIMIT 10

## Bloque 2: Resultados de calidad

Nueve corridas. Última validación de 2025: cero advertencias. Última de 2026: cinco advertencias de esquema. Todas registran cero fallos bloqueantes.
Consulta Genie: pub_toolu_bdrk_017TeLPcapKdDminXJppDuZQ_output.
Conservo el SQL ejecutado sin cambios.

In [ ]:
%sql
SELECT 
  `quality_run_id`,
  `source_feed_version`,
  `source_sha256`,
  `checked_at_utc`,
  COUNT(*) as total_checks,
  COUNT_IF(`severity` = 'BLOCKING' AND `status` = 'FAIL') as blocking_failures,
  COUNT_IF(`status` = 'WARN') as warning_checks,
  COUNT_IF(`severity` = 'WARNING' AND `status` = 'PASS') as warning_checks_passed
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_bronze_snapshot_quality`
GROUP BY `quality_run_id`, `source_feed_version`, `source_sha256`, `checked_at_utc`
ORDER BY `checked_at_utc` DESC
LIMIT 10

## Bloque 3: Catálogo completo

161 elementos y 161 versiones. El catálogo consultado se actualizó el 29 de octubre de 2026. No equivale a 161 snapshots cargados.
Consulta Genie: pub_toolu_bdrk_01FLf5sqCW2hZaU9U9WgtGD1_output.
Conservo el SQL ejecutado sin cambios.

In [ ]:
%sql
SELECT 
  COUNT(*) as total_items,
  COUNT(DISTINCT `feed_version`) as distinct_feed_versions,
  COUNT_IF(`is_direct_zip_url` = true) as direct_zip_urls,
  MIN(`feed_version`) as min_feed_version,
  MAX(`feed_version`) as max_feed_version,
  MAX(`discovered_at_utc`) as max_discovered_at_utc
FROM `workspace`.`gtfs_bogota`.`control_gtfs_static_catalog`

## Bloque 4: Inventario de encabezados

Dos parejas versión/hash, con once archivos observados cada una. Esta auditoría no certifica por sí sola una carga completa.
Consulta Genie: pub_toolu_bdrk_01UJXkM4jC19sPYmfAseEU9R_output.
Conservo el SQL ejecutado sin cambios.

In [ ]:
%sql
SELECT 
  `source_feed_version`,
  `source_sha256`,
  COUNT(DISTINCT `source_file_name`) as total_files,
  STRING_AGG(DISTINCT `source_file_name`, ', ') as file_list
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_source_schema_observations`
GROUP BY `source_feed_version`, `source_sha256`
ORDER BY `source_feed_version`

## Recuperación del planificador — 13 de noviembre de 2026

Estas seis consultas fueron verificadas mediante Genie. Solo leen auditorías y agency.
No modifican estados ni aplican resoluciones.


### Bloque 5 — Diagnóstico de intento sin cierre
Consulta verificada: pub_toolu_bdrk_018oscpXSLtk4fYHouh95PWe_output.


In [ ]:
%sql
SELECT 
  `quality_run_id`,
  `event_type`,
  `source_feed_version`,
  `source_sha256`,
  `rule_version`,
  `status`,
  `validation_completed`,
  `ready_for_silver`,
  `blocking_failures`,
  `warning_count`,
  `started_at_utc`,
  `completed_at_utc`,
  `result_sha256`
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_bronze_snapshot_runs`
WHERE `source_feed_version` IN ('2025-10-28', '2026-09-29')
  AND `source_sha256` IN ('07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952', '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
ORDER BY `started_at_utc` DESC, `event_type`
LIMIT 30

### Bloque 6 — Diagnóstico de intento sin cierre
Consulta verificada: pub_toolu_bdrk_01C7BZLcjhbwgrx8z1ft4inM_output.


In [ ]:
%sql
SELECT 
  `source_feed_version`,
  `source_sha256`,
  `silver_run_id`,
  `silver_rule_version`,
  COUNT(*) AS `filas`
FROM `workspace`.`gtfs_bogota`.`silver_gtfs_agency`
WHERE `source_feed_version` = '2025-10-28'
  AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
GROUP BY `source_feed_version`, `source_sha256`, `silver_run_id`, `silver_rule_version`
ORDER BY `silver_run_id` DESC
LIMIT 10

### Bloque 7 — Diagnóstico de intento sin cierre
Consulta verificada: pub_toolu_bdrk_01PgDtFYcJUnKzVqeH1PKdWE_output.


In [ ]:
%sql
SELECT 
  `entity`,
  `source_feed_version`,
  `source_sha256`,
  `silver_run_id`,
  `status`,
  `silver_rule_version`,
  `target_table`,
  `input_rows`,
  `candidate_rows`,
  `rejected_rows`,
  `silver_committed`,
  `started_at_utc`,
  `finished_at_utc`,
  `error_message`
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_silver_runs`
WHERE `entity` = 'agency'
  AND `source_feed_version` = '2025-10-28'
  AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
ORDER BY `started_at_utc` DESC
LIMIT 8

### Bloque 8 — Diagnóstico de intento sin cierre
Consulta verificada: pub_toolu_bdrk_01N1bZRKhmRFFkreoJFPMNRx_output.


In [ ]:
%sql
SELECT 
  `source_feed_version`,
  `source_sha256`,
  `source_file_path`,
  COUNT(*) AS `filas`
FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_agency`
WHERE `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
GROUP BY `source_feed_version`, `source_sha256`, `source_file_path`
ORDER BY `source_feed_version` DESC, `source_file_path`
LIMIT 10

### Bloque 9 — Diagnóstico de intento sin cierre
Consulta verificada: pub_toolu_bdrk_01XhNp3x7uS6prbZzapXoSAm_output.


In [ ]:
%sql
SELECT 
  `source_feed_version`,
  `source_sha256`,
  COUNT(DISTINCT `source_file_name`) AS `archivos_distintos`
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_source_schema_observations`
WHERE `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'
GROUP BY `source_feed_version`, `source_sha256`
ORDER BY `source_feed_version` DESC
LIMIT 10

### Bloque 10 — Diagnóstico de intento sin cierre
Consulta verificada: pub_toolu_bdrk_01NHysyqFxboGHEQcmnwE4qH_output.


In [ ]:
%sql
SELECT 
  COUNT(*) AS `checks_guardados`
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_bronze_snapshot_quality`
WHERE `quality_run_id` = '096d0f14-5165-43d1-9c8c-c7b0ee041221'